In [0]:
from pyspark.sql.functions import col, current_timestamp

# Define paths in Unity Catalog Volume
base_volume = "/Volumes/training_medallion/00_landing/raw_landing_zone"
landing_path = f"{base_volume}/schema_demo_landing"
checkpoint_path = f"{base_volume}/_checkpoints/schema_demo"
schema_path = f"{base_volume}/_schemas/schema_demo"

# Create landing folder
dbutils.fs.mkdirs(landing_path)

True

In [0]:


# Batch 1: Clean baseline payload
json_batch_1 = """{"ticket_id": "TCK-101", "customer_id": 100, "issue": "Payment Failed", "amount": 250.50}
{"ticket_id": "TCK-102", "customer_id": 101, "issue": "Account Locked", "amount": 120.00}"""

dbutils.fs.put(f"{landing_path}/batch_1.json", json_batch_1, overwrite=True)

# Paketkan Auto Loader dengan inferColumnTypes=true
df_batch1 = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", schema_path)
    .option("cloudFiles.inferColumnTypes", "true") # 👈 Paksa Auto Loader mendeteksi tipe numerik asli (INT/DOUBLE)
    .load(landing_path)
    .withColumn("ingestion_time", current_timestamp())
    .withColumn("source_file", col("_metadata.file_path"))
)

query1 = (
    df_batch1.writeStream
    .format("delta")
    .trigger(availableNow=True)
    .option("checkpointLocation", checkpoint_path)
    .table("training_medallion.01_bronze.bronze_schema_demo")
)
query1.awaitTermination()

Wrote 179 bytes.


26/10/29 14:54:19 Query 025cedbc-bd16-4ad0-bc1a-ea3dc655bec9 have exception: com.databricks.sql.transaction.tahoe.DeltaAnalysisExceptionWithSubErrors: [DELTA_METADATA_MISMATCH] A metadata mismatch was detected when writing to the Delta table.
- A schema mismatch detected when writing to the Delta table (Table ID: d8c772c6-69d9-4835-99a3-bb64c3eba2f0).
To enable schema migration using DataFrameWriter or DataStreamWriter, please set: '.option("mergeSchema", "true")'.
For other operations, set the session configuration spark.databricks.delta.schema.autoMerge.enabled to "true". See the documentation specific to the operation for details.

Table schema:
root
 |-- amount: double (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- issue: string (nullable = true)
 |-- ticket_id: string (nullable = true)
 |-- _rescued_data: string (nullable = true)
 |-- ingestion_time: timestamp (nullable = true)
 |-- source_file: string (nullable = true)


Data schema:
root
 |-- amount: double (nul

---------------------------------------------------------------------------
StreamingQueryException                   Traceback (most recent call last)
File <command-7666339477911315>, line 28
     10 df_batch1 = (
     11     spark.readStream
     12     .format("cloudFiles")
   (...)     18     .withColumn("source_file", col("_metadata.file_path"))
     19 )
     21 query1 = (
     22     df_batch1.writeStream
     23     .format("delta")
   (...)     26     .table("training_medallion.01_bronze.bronze_schema_demo")
     27 )
---> 28 query1.awaitTermination()

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/streaming/query.py:96, in StreamingQuery.awaitTermination(self, timeout)
     94 await_termination_cmd = pb2.StreamingQueryCommand.AwaitTerminationCommand()
     95 cmd.await_termination.CopyFrom(await_termination_cmd)
---> 96 self._execute_streaming_query_cmd(cmd)
     97 return None

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect

In [0]:
# Batch 2: Introduce malformed data type for 'customer_id' (string instead of integer)
json_batch_2 = """{"ticket_id": "TCK-103", "customer_id": "CORRUPTED_ID_999", "issue": "App Crash", "amount": 45.00}"""

dbutils.fs.put(f"{landing_path}/batch_2.json", json_batch_2, overwrite=True)

# Re-run Auto Loader pipeline
df_batch2 = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", schema_path)
    .option("cloudFiles.inferColumnTypes", "true") # 👈 Mendeteksi tipe numerik dari awal
    .option("cloudFiles.rescuedDataColumn", "_rescued_data") # 👈 Mengaktifkan penampung data bermasalah
    .load(landing_path)
    .withColumn("ingestion_time", current_timestamp())
    .withColumn("source_file", col("_metadata.file_path"))
)

(
    df_batch2.writeStream
    .format("delta")
    .trigger(availableNow=True)
    .option("checkpointLocation", checkpoint_path)
    .option("mergeSchema", "true") # 👈 Diperlukan agar Delta menyerap evolusi/perubahan skema
    .table("training_medallion.01_bronze.bronze_schema_demo")
)

In [0]:
%sql
SELECT 
  *
FROM training_medallion.01_bronze.bronze_schema_demo 

In [0]:
%sql
DROP CATALOG IF EXISTS training_medallion CASCADE;

In [0]:
%sql
-- 1. Create Training Catalog
CREATE CATALOG IF NOT EXISTS training_medallion;
USE CATALOG training_medallion;

-- 2. Create Schemas for Landing Zone & Medallion Architecture Layers
CREATE SCHEMA IF NOT EXISTS 00_landing COMMENT 'Landing Zone: Entry point for external raw files (JSON/CSV/Logs)';
CREATE SCHEMA IF NOT EXISTS 01_bronze  COMMENT 'Bronze Layer: Raw ingested Delta tables from Auto Loader';
CREATE SCHEMA IF NOT EXISTS 02_silver  COMMENT 'Silver Layer: Cleansed, deduplicated, and validated data';
CREATE SCHEMA IF NOT EXISTS 03_gold    COMMENT 'Gold Layer: Business-ready aggregated data marts';

-- 3. Create Unity Catalog Volume in the Landing Zone
CREATE VOLUME IF NOT EXISTS 00_landing.raw_landing_zone
COMMENT 'Unity Catalog Volume for receiving incoming raw files before ingestion';

In [0]:
# 1. Hapus tabel Delta lama
spark.sql("DROP TABLE IF EXISTS training_medallion.01_bronze.bronze_schema_demo")

# 2. Hapus checkpoint, schema, dan folder landing agar tidak ada metadata tersisa
dbutils.fs.rm(checkpoint_path, recurse=True)
dbutils.fs.rm(schema_path, recurse=True)
dbutils.fs.rm(landing_path, recurse=True)

# 3. Buat ulang folder landing
dbutils.fs.mkdirs(landing_path)
print("🧹 Workspace bersih untuk pengujian ulang!")

🧹 Workspace bersih untuk pengujian ulang!


In [0]:
# Batch 3: Introduce new attributes ('priority' and 'region')
json_batch_3 = """{"ticket_id": "TCK-104", "customer_id": 103, "issue": "Refund Query", "amount": 310.00, "priority": "HIGH", "region": "APAC"}"""

dbutils.fs.put(f"{landing_path}/batch_3.json", json_batch_3, overwrite=True)

# Read stream with Auto Loader schema evolution enabled
df_batch3 = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", schema_path)
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns") # Instruct Auto Loader to evolve schema upon detecting new fields
    .load(landing_path)
    .withColumn("ingestion_time", current_timestamp())
    .withColumn("source_file", col("_metadata.file_path"))
)

# Write stream to Delta Lake with schema merging enabled
query3 = (
    df_batch3.writeStream
    .format("delta")
    .trigger(availableNow=True)
    .option("checkpointLocation", checkpoint_path)
    .option("mergeSchema", "true") # Delta Lake option to dynamically alter target table schema
    .table("training_medallion.01_bronze.bronze_schema_demo")
)

query3.awaitTermination()

Wrote 125 bytes.


26/10/29 14:54:32 Listener thread still active, was not able to reconnect
26/10/29 14:54:34 Query 3040b8b2-8ed4-4838-aeff-9b5f8f49c65c have exception: org.apache.spark.SparkNumberFormatException: [CAST_INVALID_INPUT] The value 'CORRUPTED_ID_999' of the type "STRING" cannot be cast to "BIGINT" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018
	at org.apache.spark.sql.errors.QueryExecutionErrors$.invalidInputInCastToNumberError(QueryExecutionErrors.scala:205)
	at com.databricks.photon.PhotonException$.getSparkException(PhotonException.scala:305)
	at com.databricks.photon.PhotonWriteResultHandler.getResult(PhotonWriteStageExec.scala:153)
	at com.databricks.photon.PhotonBasicEvaluatorFactory$PhotonBasicEvaluator$$anon$1.open(PhotonBasicEvaluatorFactory.scala:299)
	at com.databricks.photon.PhotonBasicEvaluatorFactory$PhotonBasicEvaluator$$anon$1.hasNextImpl(PhotonBasicE

---------------------------------------------------------------------------
StreamingQueryException                   Traceback (most recent call last)
File <command-4749707821044865>, line 28
     18 # Write stream to Delta Lake with schema merging enabled
     19 query3 = (
     20     df_batch3.writeStream
     21     .format("delta")
   (...)     25     .table("training_medallion.01_bronze.bronze_schema_demo")
     26 )
---> 28 query3.awaitTermination()

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/streaming/query.py:96, in StreamingQuery.awaitTermination(self, timeout)
     94 await_termination_cmd = pb2.StreamingQueryCommand.AwaitTerminationCommand()
     95 cmd.await_termination.CopyFrom(await_termination_cmd)
---> 96 self._execute_streaming_query_cmd(cmd)
     97 return None

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/streaming/query.py:184, in StreamingQuery._execute_streaming_query_cmd(self, cmd)
    182 exec_cmd = pb2